
## Predictions

We predict how each time grows from n = 5,000 to n = 25,000: O(1) → about 1×, O(log n) → about 1.2×, O(n) → about 5×, O(n²) → about 25×.

### Linked list
Our linked list stores only head, so count() walks the whole list, every index-based method calls count() to check its index, and every insert calls findById to reject duplicate IDs. Almost every operation is therefore O(n), even when the change itself only relinks one node.

| Operation | Nodes visited (about) | Predicted | 25k / 5k |
|---|---|---|---|
| Read middle | `count()` n + walk to the middle n/2 = 1.5n | O(n) | ≈5 |
| Insert at beginning | duplicate check n, then relink the head | O(n) | ≈5 |
| Append | duplicate check n + walk to the last node n = 2n | O(n) | ≈5 |
| Insert in middle | `count()` n + duplicate check n + walk n/2 = 2.5n | O(n) | ≈5 |
| Remove from beginning | `count()` n, then relink the head | O(n) | ≈5 |
| Remove from middle | `count()` n + walk n/2 = 1.5n | O(n) | ≈5 |
| Search for missing ID | every node: n | O(n) | ≈5 |
| Linear search (middle ID) | n/2 | O(n) | ≈5 |

At the same n, we expect insert in middle to be the slowest of these and the linear search for the middle ID the fastest. Removing from the beginning is the interesting case: unlinking the head is O(1), but the index check calls count() first, so we predict it grows like O(n). With a cached size it would be O(1).

Binary search needs the element at mid on every step. On our list that is getAt(mid), which costs O(n) (count() plus a walk), so one search takes O(log n) steps × O(n) each = O(n log n): slower than a plain linear search.

Insertion sort by priority: Our insertion sort builds a new sorted chain. Each node walks from the front of that chain past every node whose priority is less than or equal to its own.
- Sorted input: worst case - Each new node has the largest priority so far, so it walks to the end of the chain: about n²/2 steps. This is the opposite of the textbook array version, where sorted input is the best case.
- Reverse-sorted input: fastest - A node with a smaller priority than everything so far goes straight to the front. But with only 4 priority values, a node must still walk past the earlier nodes with the same priority: about n²/8 steps.
- Random input: each node walks past the nodes with priority ≤ its own, on average 5/8 of the chain: about 0.31 n² steps.

All three are O(n²), so doubling n (1,000 → 2,000) should take about 4× as long. We expect sorted input to be the slowest and reverse-sorted about 4× faster than sorted.

### Array
_Assuming the final version checks for duplicate IDs with `findById` (a scan of the whole array) before every insert, as the linked list does._

The array keeps its size in a field, so count() and the index checks are O(1), and reading any position is a single list lookup. Inserting or removing shifts every element after that position by one slot. Every insert also scans the array for a duplicate ID first.

| Operation | Elements touched (about) | Predicted | 25k / 5k |
|---|---|---|---|
| Read middle | 1 | O(1) | ≈1 |
| Insert at beginning | duplicate check n + shift n = 2n | O(n) | ≈5 |
| Append | duplicate check n, then write one slot | O(n) | ≈5 |
| Insert in middle | duplicate check n + shift n/2 = 1.5n | O(n) | ≈5 |
| Remove from beginning | shift n | O(n) | ≈5 |
| Remove from middle | shift n/2 | O(n) | ≈5 |
| Search for missing ID | n | O(n) | ≈5 |
| Linear search (middle ID) | n/2 | O(n) | ≈5 |
| Binary search | about log₂ n (12 to 15 steps) | O(log n) | ≈1.2 |

For append, writing into the next free slot is O(1) (amortized over resizes), but the duplicate check makes it O(n). Without the check it would be ≈1.

Compared with the linked list: reading the middle should be far faster on the array (O(1) against O(n)). For the inserts, both structures are dominated by the same duplicate scan, so we expect them to be within a small factor of each other. Removing from the beginning is O(n) on both, for different reasons: the array shifts every element, while the linked list counts every node.

Insertion sort by priority: The array's insertion sort re-inserts the requests one at a time into an emptied array. For each request it scans from the front to the first element with a larger priority, then insertAt shifts everything after that point one slot right. The scan and the shift together always cover the whole current array, so inserting the i-th request costs about i steps whatever the input order, plus about i more for the duplicate check. That is about n² steps for sorted, reverse-sorted and random input alike: O(n²) every time, with no best case. Doubling n should take about 4× as long, and all three input orders should take about the same time.


### Workload (Part K, Profile D: audit and cleanup)
Starting from n = 10,000 requests, 20,000 operations: 40% findById (80% existing IDs, 20% missing), 30% insertAt(middle), 20% removeAt(middle), 10% getAt(random index). That is about 6,000 inserts and 4,000 removals, so the collection should end at about 12,000 requests (about 11,000 on average during the run). The array's capacity at that size is 16,384, so we expect no resizing during the replay.

Work per operation at size n:

| Operation | Share | Array | Linked list |
|---|---|---|---|
| findById | 40% | 0.6n (a hit stops halfway on average, a miss checks all n) | 0.6n (same) |
| insertAt(middle) | 30% | duplicate check n + shift n/2 = 1.5n | `count()` n + duplicate check n + walk n/2 = 2.5n |
| removeAt(middle) | 20% | shift n/2 = 0.5n | `count()` n + walk n/2 = 1.5n |
| getAt(random) | 10% | 1 | `count()` n + walk n/2 on average = 1.5n |
| Weighted average | | about 0.8n | about 1.4n |

Dominant operations: findById is the most frequent, but insertAt(middle) costs the most: it is over half of the work on both structures, and its duplicate check alone is a large part of that. Every operation except the array's getAt is O(n), so the whole replay is O(operations × n) on both.

Prediction: The array should finish the replay faster, roughly 1.5–2×, because it never walks to reach a position (getAt is O(1)), and removing from the middle only shifts half the elements, instead of counting every node and then walking to the middle.

A change that could reverse this: If the workload mostly added and removed at the front (like Profile C) and the linked list kept a cached size, removeAt(0) would be O(1) on the linked list but still O(n) on the array, since every element shifts. The linked list could then come out ahead.

## Part H — Complexity Analysis

### Summary — Linked list (worst-case)

| Operation | Time | Temporary working space |
|---|---|---|
| Indexed access (getAt) | O(n) | O(1) |
| Linear Search (findById) | O(n) | O(1) |
| Push / prepend | O(n) | O(1) |
| Append | O(n) | O(1) |
| Insertion at arbitrary position (insertAt) | O(n) | O(1) |
| Removal at arbitrary position (removeAt) | O(n) | O(1) |
| Count | O(n) | O(1) |
| Linear search (findById) | O(n) | O(1) |
| Insertion sort | O(n^2) | O(1) |
| Merge sort | O(n log n) | O(log n) |


### Summary — Array (worst-case)

| Operation | Time | Temporary working space |
|---|---|---|
| Indexed access (getAt) | O(1) | O(1) |
| Search (findById) | O(n) | O(1) |
| Push (insert at front) | O(n) | O(1), O(n) during a resize |
| Append | O(n) worst case (resize), O(1) amortized | O(1), O(n) during a resize |
| Insertion at arbitrary position (insertAt) | O(n) | O(1), O(n) during a resize |
| Removal at arbitrary position (removeAt) | O(n) | O(1), O(n) during a shrink |
| Count | O(1) | O(1) |
| Linear search (findById) | O(n) | O(1) |
| Binary search | O(log n) | O(1) |
| Insertion sort | O(n^2) | O(n) |
| Merge sort | O(n log n) | O(n) |

---

### Linked list breakdown

**Indexed access — getAt:**
Indexed access time complexity is O(n) because getAt returns the request at a given position. First it does a boundaries check to make sure that the index is an integer, which is O(1), then it checks if the integer is in range by calling count(). Because the size is not stored, count has to walk the entirety of the list, which is O(n). Because count can take up to n steps, time complexity is O(n). Temporary working space is O(1) because it uses two variables that don't change no matter how long the list is.

**Linear Search — findById:**
Linear search is O(n). findById looks for a request by the ID and returns its position or -1 if the position is not found. It starts at the head and walks through the list comparing each node's request_id with the ID that is being searched for. In the worst case, the ID does not exist in the list, so it would have to check every n node before returning -1. Because of this, time complexity is O(n). Temporary working space is O(1) because it only uses two variables that do not change no matter how long the list is.

**Push / prepend:**
Push is O(n). Before push adds a new request to the front of the list, it checks to make sure that the request is not already in the list. To confirm the ID is not already in the list, findById has to walk the whole list to confirm, which is O(n). If the check passes, push creates a new node, points it at the current head, and makes it the new head. This step is just O(1), because it is only 2 changes, it is checking for duplicates that makes push O(n). It only uses one variable while it runs(tempNode), so the temporary working space is O(1). 

**Append:**
Append is O(n). Before append adds a new request to the back of the list, it checks to make sure that the request is not already in the list by calling findbyId for the duplicate check which as we know is O(n). Then, because there is no tail pointer, append has to walk n steps until it finds where to attach the new node which is also O(n). That is two full walks back to back, but big O drops constants so the notation remains as O(n). Temporary working space is O(1) because it only uses two variables(prevNode and lastNode), and these do not change as the size of n grows. 

**Insertion at arbitrary position — insertAt:**
insertAt is O(n). insertAt first has to do a boundary check to make sure the request is an integer -O(1),  and that is in range - O(n). To check if the integer is in range, it has to call count() which walks the whole list making it O(n). Next, findById is called for the duplicate check which is another O(n). Then, if the index is 0, it hands off the insertion to push, which is also O(n). If index is not 0, it walks from the head node to the node just before the insertion point which is n steps. Splicing the node is just two arrow changes, so that is O(1). So insertAt is up to 3n, but because Big O notation drops constants, its notation is O(n). The temporary working space is O(1) because it uses only 3 variables that do not grow as n grows.

**Removal at arbitrary position — removeAt:.**
removeAt is O(n). First, removeAt does boundary checks to make sure that the request is an integer and that the request is in range. Checking if the request is an integer is O(1) and checking the range is O(n) because count() has to be called. Then, the program checks if the index is 0, in this case it would just move the head to the next node which is O(1). Otherwise, removeAt walks from the head to one before the node that is being removed which takes up to n steps. Removing the node is one arrow change - O(1). The walks are what make removeAt O(n). removeAt only uses a few variables, and the number of them doesn’t grow as n grows, so temporary space is O(1).

**Count - count():**
count() is O(n). Because the size is not stored, count() has to start at the head node and walk its way through every node, making it O(n). Not using a cached size is a tradeoff I chose to make for simplicity. But I am paying for it as count() is O(n) instead of O(1). And everything that calls count() is also paying for it through an extra walk. Count() only uses two variables as it runs, and the number of them does not grow with n, so the temporary working space is O(1).

**Binary search — why it doesn't suit a linked list:**
Binary search works through repeatedly checking the middle of a sorted range and discarding the half that cannot contain the value being searched for. Binary search relies on being able to jump directly to the middle position and unfortunately, a linked list does not have direct access to a position. If we wanted to reach the middle of a linked list, we would have to walk node by node until we got there from the head which is O(n). This compiling of walks would make binary search O(nlogn), which is slower than a simple linear search at O(n). For this reason, using a linear search simply makes more sense than a binary search for linked lists.

**Insertion sort by priority:**
The insertion sort takes nodes one at a time off of the original list and places each one into the correct spot on a new sorted chain. The best case scenario is O(n). This would occur when the nodes are already sorted in strictly decreasing order (no repeated priorities). In this case, every node would be smaller than the current head of the sorted chain, so, it would just go straight to the front. The worst case scenario would be O(n^2). This would happen if the list was already sorted in increasing order. In this case, every node would belong at the end so the walker would have to go through the entire chain each time. Those walks add up to n^2 steps. The average case would also be O(n^2) because with random input, on average, each node would walk halfway through the sorted chain. Additionally, because priority only has 4 levels, large lists will have many of the same priorities, and since the sort keeps equal priorities in their original order, each node walks past every node with the same priority. So in practice, large inputs behave closer to O(n^2) even when reverse sorted. The temporary working space is O(1). It only rearranges the arrows of existing nodes and doesn’t create new nodes or lists, and the number of variables does not grow with n.

**Merge sort by reported_time:**
The merge sort is quite a bit more complicated than the insertion sort. The merge sort works by splitting the list in half recursively until every piece is just a single node. Then the pieces are merged together as the recursive calls return. Merge sort is O(nlogn) for the best case, worst case, and average case. This is because merge sort always splits the list in half in the same way regardless of what the values are. The list is cut in half log n times. For each chain, splitMiddle walks through it with the slow and fast walkers, and mergeByTime attaches each node while combining the chains. So each level is O(n) and there are logn levels making the complexity O(nlogn). The temporary working space is O(logn). If it were not for the recursion, the temporary working space would be O(1) as the variables only change the arrows of the existing nodes. But, because of recursion, extra memory is needed for the calls of sortChainByTime within itself.

---

### Array breakdown

**Indexed access:**
getAt is O(1). getAt checks if the index is greater than or equal to its current size which is O(1) as it is one comparison. It then returns the element that is in that position, which is O(1). An array stores elements in consecutive positions, so any index is able to be reached directly, therefore, nothing is walked or shifted, so getAt complexity is O(1). Temporary working space is also O(1) as none of the variables grow with n.

**Linear Search — findById:**
findById is O(n). It first checks to see if the array is empty which is O(1). It checks each position from index 0 up to the size of the array. If the request is not found it returns -1. It compares each request's ID to the one that is being searched for and returns the position once a match is found. If the ID is not in the array, it has to check every position leading to time complexity being O(n). The temporary working space is O(1) as variables are not growing with n.

**Push:**
push is O(n). To put a request at index 0, every element has to be shifted over by one slot to the right to make room for the new request. Shifting happens everytime push is called so the complexity would be O(n). The new request is placed at index 0 which is O(1). If the array is full, push has to resize which is O(n). Temporary working space is O(1) normally, but, during a resize it is O(n) as a new buffer has to be created and every element in the array has to be copied over.

**Append:**
append is O(1) on average over many appends, and O(n) in the worst case, when resizing is required. The array knows where the next open slot is because it knows its size, so adding to the end is O(1). No searching or shifting is required. However, if the array is full, it has to resize and copy all elements over to a new buigger array, which is O(n). Because the capacity doubles each time, resizes happen less and less often as the array grows, so when the cost is spread out over many appends, it averages out to O(1) per append. Temporary working space is O(1) normally and O(n) during a resize.

**Insertion at arbitrary position — insertAt:**
insertAt is O(n). insertAt first checks if the index is nonnegative and not past the current size. These checks are O(1). If the array is full, it resizes by copying over everything into a new bigger array which is O(n). Then, to make room for the new request, it shifts every element from the index to the end over by one slot to the right. O(n). Then, the insertion of the new request into the open slot is just O(1). Shifting however dominates so the complexity of insertAt is O(n). Temporary working space is O(1), unless a resize occurs which makes it O(n).

**Removal at arbitrary position — removeAt:**
removeAt is O(n). removeAt finds the request by its ID so a linear scan is needed to find the matching ID. Once found and removed, every element after it shifts over one slot to the left to close the gap. Therefore, because a linear search is required with up to n steps, the complexity is O(n). The program then checks if the new size of the array is a power of 2 which is O(logn) because it keeps halving the current size if it is as well as if the current size is greater than 7. The array shrinks to half of its capacity if it can by copying all the elements into a new smaller array which is O(n). Temporary working space is O(1) normally and O(n) during a shrink.

**Count:**
count() is O(1). The array keeps track of its size, so count does not need to search the array. It simply just returns its size in one step. Temporary working space is also O(1) because it doesn't use any extra variables.

**Binary search by ID:**
Binary search is O(logn). It keeps a range using a lower and i[[er index starting at the first and last positions. At each step, it checks the middle element of the range. If it matches, it returns that position. If the middle position ID is too big or too small, it throws out the half that cannot contain the ID. A range of elements can only be halved logn times making the notation O(logn). Temporary working space is O(1) as the present variables do not change as the size of n grows.]

**Insertion sort by priority:**
Insertion sort is O(n^2). Insertion sort copies elements of the array into a new array by adding back the elements in order one at a time. It scans from front to back to find where the request goes, then everything to the right of the element shifts right to make room for the request. Because n elements has to be scanned or shifted n times, which results in n steps n times, the resulting notation is O(n^2). Temporary working space is O(n) because it has to copy the whole array.

**Merge sort by reported time**
Merge sort is O(nlogn). Merge sort works by copying the array into the new list and then recursively splitting the array in half until each array is just one element. Then, merges the elements back together in order of the reported time. The recursion part is O(logn) and at each level of recursion, O(n) work occurs when the respective arrays are split or merged. So therefore, the notation is O(nlogn). The temporary working space is O(n) because new arrays are made each time the array splits.

## Part G — Performance Experiment
### Experimental setup

Sizes: n = 1,000, 5,000, 10,000 and 25,000 for the single operations. Insertion sort uses n = 1,000, 2,000, 5,000 and 10,000 instead: it is O(n²), so one sort at 25,000 is hundreds of millions of steps, and 5 runs × 3 input orders × 2 structures would take hours.

Same data for both structures: For each n, generate_requests(n) produces one list of requests, and both structures are built from that list with append, in the same order.

Timing: time.perf_counter() is read immediately before and after the operation, so only the operation itself is timed. Each result is the median of 5 runs, so one run slowed down by the operating system does not distort it. Operations that change nothing can be faster than the timer measures reliably: timing an empty function once gave 1.7 µs, but timing 10,000 calls together gave 0.08 µs per call. So each run times 100 of these calls in a row and divides by 100.

Build once, undo every change: Building the linked list costs about n² steps, because each append first checks for a duplicate ID and then walks to the last node; at n = 10,000 the build alone takes about 6 seconds. So each structure is built once per size and reused for every operation. Each timed change is undone straight away (not timed), so every run sees exactly n requests:

| Operation | Timed call | Undo (not timed) |
|---|---|---|
| Read middle | `getAt(n // 2)` | — |
| Insert at beginning | `push(new)` | `removeAt(0)` |
| Append | `append(new)` | `removeAt(n)` |
| Insert in middle | `insertAt(n // 2, new)` | `removeAt(n // 2)` |
| Remove from beginning | `removeAt(0)` | put the request back at 0 |
| Remove from middle | `removeAt(n // 2)` | put the request back at n // 2 |
| Search for missing ID | `findById(998)` | — |
| Linear search | `findById` of the middle request's ID | — |

The new request has ID 999 and the missing ID is 998. Generated IDs start at 1000, so the new request is never a duplicate and 998 is never found. Searching for the missing ID is the worst case (every request is checked); searching for the middle request's ID is the typical case (the search stops halfway).

Array resizing:  After building, the array's capacity is 1,024, 8,192, 16,384 and 32,768 (always a power of two). Every n is below its capacity, so inserting one request never fills the array; and neither n nor n − 1 is a power of two, so removing one never triggers a shrink. The array times therefore show the cost of shifting elements, not of copying into a new buffer.

Binary search (array only): A second array holds the same requests in increasing ID order (sorted() is used only to prepare this input). Each run searches for about 100 IDs spread evenly across the array, and the total is divided by the number of searches. A single ID would mislead: the exact middle ID is found on the first step. The linked list is not timed for binary search; the spec asks for an analysis instead (Part H).

Insertion sort: Three orders of the same requests: sorted by priority, reverse-sorted, and the dataset's random order. Priority has only four values, so every input contains many ties. Sorting changes the order, so a fresh structure is built (not timed) before each of the 5 runs.

Workload (Part K): _To be added once `workload.py` is ready._

Note: Early test runs on unfinished code happened before the predictions were written. Their results were discarded and are not used anywhere in this notebook.

In [ ]:
import statistics
import time

import matplotlib.pyplot as plt
import pandas as pd
from matplotlib.ticker import LogFormatter, NullLocator

from maintenance_request import MaintenanceRequest, generate_requests
from array_request_list import ArrayRequestList
from linked_request_list_sort import LinkedRequestList

SIZES = [1_000, 5_000, 10_000, 25_000]
RUNS = 5


def time_once(operation, repeat=1):
    """Seconds for one call of operation(). Very fast operations are run
    `repeat` times in a row and the total is divided by repeat."""
    start = time.perf_counter()
    for _ in range(repeat):
        operation()
    return (time.perf_counter() - start) / repeat


def median_time(operation, undo=None, repeat=1, runs=RUNS):
    """Median of `runs` timings. If undo is given, it runs after each
    timing (not timed) to put the structure back the way it was."""
    times = []
    for _ in range(runs):
        times.append(time_once(operation, repeat))
        if undo is not None:
            undo()
    return statistics.median(times)


In [ ]:
def build(structure_class, requests):
    """A new structure holding these requests in this order (built with append)."""
    s = structure_class()
    for r in requests:
        s.append(r)
    return s


def ids(structure):
    """The IDs in order, read through count() and getAt()."""
    return [structure.getAt(i).request_id for i in range(structure.count())]

In [ ]:
NEW = MaintenanceRequest(999, "CASE", "HVAC", 3, 0)   # ID below 1000, so never in the dataset
MISSING_ID = 998                                       # also never in the dataset
READ_REPEAT = 100


def time_operations(s):
    """Median time of each operation on structure s. Every change is undone
    straight after it is timed, so s ends up exactly as it started."""
    n = s.count()
    middle = n // 2
    first = s.getAt(0)
    middle_request = s.getAt(middle)
    return {
        "read middle": median_time(lambda: s.getAt(middle), repeat=READ_REPEAT),
        "insert at beginning": median_time(lambda: s.push(NEW), undo=lambda: s.removeAt(0)),
        "append": median_time(lambda: s.append(NEW), undo=lambda: s.removeAt(n)),
        "insert in middle": median_time(lambda: s.insertAt(middle, NEW), undo=lambda: s.removeAt(middle)),
        "remove from beginning": median_time(lambda: s.removeAt(0), undo=lambda: s.insertAt(0, first)),
        "remove from middle": median_time(lambda: s.removeAt(middle), undo=lambda: s.insertAt(middle, middle_request)),
        "search missing ID": median_time(lambda: s.findById(MISSING_ID), repeat=READ_REPEAT),
        "linear search": median_time(lambda: s.findById(middle_request.request_id), repeat=READ_REPEAT),
    }

In [ ]:
SORT_SIZES = [1_000, 2_000, 5_000, 10_000]


def time_binary_search(by_id):
    """Array only. Time per findById_Binary call, averaged over about 100 IDs
    spread across the array (which must be in increasing ID order)."""
    n = by_id.count()
    targets = [by_id.getAt(i).request_id for i in range(0, n, max(1, n // 100))]

    def search_all():
        for request_id in targets:
            by_id.findById_Binary(request_id)

    return median_time(search_all) / len(targets)


def insertion_sort_inputs(n):
    """The same n requests in three orders by priority."""
    requests = generate_requests(n)
    by_priority = sorted(requests, key=lambda r: r.priority)   # only prepares the input
    return {
        "sorted": by_priority,
        "reverse-sorted": by_priority[::-1],
        "random": requests,
    }


def time_insertion_sort(structure_class, requests, sort):
    """Median time to sort a fresh structure built from requests.
    Sorting changes the order, so it is rebuilt (not timed) before every run."""
    times = []
    for _ in range(RUNS):
        s = build(structure_class, requests)
        times.append(time_once(lambda: sort(s)))
    return statistics.median(times)

In [ ]:
STRUCTURES = {"array": ArrayRequestList, "linked list": LinkedRequestList}
SORTS = {"array": lambda s: s.sort_by_priority(), "linked list": lambda s: s.insertionSortPriority()}

results = []
for n in SIZES:
    requests = generate_requests(n)
    for structure, structure_class in STRUCTURES.items():
        s = build(structure_class, requests)
        for operation, seconds in time_operations(s).items():
            results.append({"operation": operation, "structure": structure, "n": n, "seconds": seconds})
    by_id = build(ArrayRequestList, sorted(requests, key=lambda r: r.request_id))
    results.append({"operation": "binary search", "structure": "array", "n": n,
                    "seconds": time_binary_search(by_id)})
    print(f"operations: n = {n:,} done")

sort_results = []
for n in SORT_SIZES:
    for order, requests in insertion_sort_inputs(n).items():
        for structure, structure_class in STRUCTURES.items():
            seconds = time_insertion_sort(structure_class, requests, SORTS[structure])
            sort_results.append({"structure": structure, "order": order, "n": n, "seconds": seconds})
    print(f"insertion sort: n = {n:,} done")

In [ ]:
ops = pd.DataFrame(results)
ops_table = ops.pivot_table(index=["operation", "structure"], columns="n", values="seconds", sort=False) * 1e6
ops_table["25k / 5k"] = ops_table[25_000] / ops_table[5_000]
ops_table.round(2)


In [ ]:
sorts = pd.DataFrame(sort_results)
sort_table = sorts.pivot_table(index=["order", "structure"], columns="n", values="seconds", sort=False)
sort_table["2k / 1k"] = sort_table[2_000] / sort_table[1_000]
sort_table.round(3)

In [ ]:

COLORS = {"array": "#2a78d6", "linked list": "#eb6834"}
plt.rcParams.update({
    "figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb",
    "axes.edgecolor": "#c3c2b7", "axes.labelcolor": "#52514e", "text.color": "#0b0b0b",
    "xtick.color": "#52514e", "ytick.color": "#52514e",
    "axes.grid": True, "grid.color": "#e6e5e0", "grid.linewidth": 0.8,
    "axes.spines.top": False, "axes.spines.right": False,
    "lines.linewidth": 2, "lines.markersize": 6,
})

fig, axes = plt.subplots(3, 3, figsize=(13, 10))
for ax, operation in zip(axes.flat, ops["operation"].unique()):
    for structure, color in COLORS.items():
        rows = ops[(ops["operation"] == operation) & (ops["structure"] == structure)]
        if len(rows) > 0:
            ax.plot(rows["n"], rows["seconds"] * 1e6, marker="o", color=color, label=structure)
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xticks(SIZES, [f"{n // 1000}k" for n in SIZES])
    ax.xaxis.set_minor_locator(NullLocator())
    ax.yaxis.set_major_formatter(LogFormatter())
    ax.yaxis.set_minor_formatter(LogFormatter(labelOnlyBase=False, minor_thresholds=(1, 0.4)))
    ax.set_title(operation)
    ax.set_xlabel("n (requests)")
    ax.set_ylabel("median time (µs)")
fig.legend(*axes.flat[0].get_legend_handles_labels(), loc="upper center", ncol=2, frameon=False)
fig.tight_layout(rect=(0, 0, 1, 0.96))
plt.show()

In [ ]:

fig, axes = plt.subplots(1, 3, figsize=(13, 4.5), sharey=True)
for ax, order in zip(axes, ["sorted", "reverse-sorted", "random"]):
    for structure, color in COLORS.items():
        rows = sorts[(sorts["order"] == order) & (sorts["structure"] == structure)]
        ax.plot(rows["n"], rows["seconds"], marker="o", color=color, label=structure)
    ax.set_title(f"{order} input")
    ax.set_xlabel("n (requests)")
axes[0].set_ylabel("median time (s)")
fig.legend(*axes[0].get_legend_handles_labels(), loc="upper center", ncol=2, frameon=False)
fig.tight_layout(rect=(0, 0, 1, 0.9))
plt.show()